## EXERCICE 05 FICHE DE TD INF 4117

### Application avec GSP

In [55]:
import pandas as pd
import subprocess
import re
import math

# =======================
# 1️⃣ Jeu de données
# =======================
data = {
    "S.ID": [1, 2, 3, 4, 5],
    "Sequence": [
        "<{1 5}{2}{3}{4}>",
        "<{1}{3}{4}{3 5}>",
        "<{1}{2}{3}{4}>",
        "<{1}{3}{5}>",
        "<{4}{5}>"
    ]
}

df = pd.DataFrame(data)
print("=== Jeu de données initial ===")
print(df, "\n")

# =======================
# 2️⃣ Conversion correcte en format SPMF
# =======================
def convert_to_spmf(seq_str):
    seq_str = seq_str.strip()[1:-1]  # retire les < >
    itemsets = re.findall(r"\{([^}]*)\}", seq_str)
    spmf_seq = " -1 ".join(itemsets) + " -2"
    return spmf_seq

spmf_sequences = [convert_to_spmf(s) for s in df["Sequence"]]
dataset_str = "\n".join(spmf_sequences)

# Sauvegarde dans le format attendu par SPMF
with open("dataset.txt", "w") as f:
    f.write(dataset_str + "\n")

print("=== Format SPMF généré ===")
print(dataset_str, "\n")

# =======================
# 3️⃣ Exécution du GSP avec support = 33%
# =======================
minsup_percent = 33
support_abs = math.ceil(len(df) * minsup_percent / 100)
print(f"Support minimal = {minsup_percent}% ({support_abs} séquences sur {len(df)})\n")

command = [
    "java", "-jar", "spmf.jar",
    "run", "GSP",
    "dataset.txt", "output.txt",
    f"{minsup_percent}%"
]

print("Exécution de l’algorithme GSP...\n")
subprocess.run(command)

# =======================
# 4️⃣ Lecture et affichage propre des résultats
# =======================
clean_results = []
with open("output.txt", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Exemple: "1 -1 3 -1 4 #SUP: 3"
        parts = line.split("#SUP:")
        seq_part = parts[0].strip()
        support = parts[1].strip() if len(parts) > 1 else "?"
        # Nettoyer la séquence
        items = [s.strip() for s in seq_part.split("-1") if s.strip()]
        seq_readable = "".join(f"<{i.strip()}>" for i in items)
        clean_results.append(f"{seq_readable}  (support: {support})")

# =======================
# 5️⃣ Affichage final propre
# =======================
if clean_results:
    print("=== Résultats GSP (propres) ===")
    for seq in clean_results:
        print(seq)
else:
    print("Aucun motif fréquent trouvé.")


=== Jeu de données initial ===
   S.ID          Sequence
0     1  <{1 5}{2}{3}{4}>
1     2  <{1}{3}{4}{3 5}>
2     3    <{1}{2}{3}{4}>
3     4       <{1}{3}{5}>
4     5          <{4}{5}> 

=== Format SPMF généré ===
1 5 -1 2 -1 3 -1 4 -2
1 -1 3 -1 4 -1 3 5 -2
1 -1 2 -1 3 -1 4 -2
1 -1 3 -1 5 -2
4 -1 5 -2 

Support minimal = 33% (2 séquences sur 5)

Exécution de l’algorithme GSP...

>/home/mlee/spmf.jar
=============  Algorithm - STATISTICS =============
 Total time ~ 12 ms
 Frequent sequences count : 9
 Max memory (mb):10.993026733398438

=== Résultats GSP (propres) ===
<1>  (support: 4)
<2>  (support: 2)
<3>  (support: 4)
<4>  (support: 4)
<5>  (support: 4)
<1><2>  (support: 2)
<1><3>  (support: 4)
<2><3>  (support: 2)
<1><2><3>  (support: 2)


### Application avec SPADE

In [57]:
import pandas as pd
import subprocess
import re

# =======================
# 1️⃣ Jeu de données
# =======================
data = {
    "S.ID": [1, 2, 3, 4, 5],
    "Sequence": [
        "<{1 5}{2}{3}{4}>",
        "<{1}{3}{4}{3 5}>",
        "<{1}{2}{3}{4}>",
        "<{1}{3}{5}>",
        "<{4}{5}>"
    ]
}

df = pd.DataFrame(data)
print("=== Jeu de données initial ===")
print(df, "\n")

# =======================
# 2️⃣ Conversion correcte en format SPMF
# =======================
def convert_to_spmf(seq_str):
    # Supprimer les chevrons extérieurs
    seq_str = seq_str.strip()[1:-1]
    # Séparer les itemsets
    itemsets = re.findall(r"\{([^}]*)\}", seq_str)
    # Convertir en format SPMF : items séparés par espace, -1 entre itemsets, -2 à la fin
    spmf_seq = " -1 ".join(itemsets) + " -2"
    return spmf_seq

spmf_sequences = [convert_to_spmf(s) for s in df["Sequence"]]
dataset_str = "\n".join(spmf_sequences)

# Écriture du fichier dataset.txt
with open("dataset.txt", "w") as f:
    f.write(dataset_str + "\n")

print("=== Format SPMF généré ===")
print(dataset_str)
print()

# =======================
# 3️⃣ Exécution du GSP (minSup = 33%)
# =======================
command = [
    "java", "-jar", "spmf.jar",
    "run", "SPADE",
    "dataset.txt", "output_SPADE.txt",
    "33%"
]

print("Exécution de l’algorithme GSP...")
subprocess.run(command)

# =======================
# 4️⃣ Lecture et affichage des résultats
# =======================
with open("output_SPADE.txt", "r") as f:
    results = f.read()

print("=== Résultats de SPADE ===")
print(results if results else "Aucun motif fréquent trouvé.")
# =======================
# 4️⃣ Lecture et affichage propre des résultats
# =======================
clean_results = []
with open("output_SPADE.txt", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Exemple: "1 -1 3 -1 4 #SUP: 3"
        parts = line.split("#SUP:")
        seq_part = parts[0].strip()
        support = parts[1].strip() if len(parts) > 1 else "?"
        # Nettoyer la séquence
        items = [s.strip() for s in seq_part.split("-1") if s.strip()]
        seq_readable = "".join(f"<{i.strip()}>" for i in items)
        clean_results.append(f"{seq_readable}  (support: {support})")

# =======================
# 5️⃣ Affichage final propre
# =======================
if clean_results:
    print("=== Résultats GSP (propres) ===")
    for seq in clean_results:
        print(seq)
else:
    print("Aucun motif fréquent trouvé.")


=== Jeu de données initial ===
   S.ID          Sequence
0     1  <{1 5}{2}{3}{4}>
1     2  <{1}{3}{4}{3 5}>
2     3    <{1}{2}{3}{4}>
3     4       <{1}{3}{5}>
4     5          <{4}{5}> 

=== Format SPMF généré ===
1 5 -1 2 -1 3 -1 4 -2
1 -1 3 -1 4 -1 3 5 -2
1 -1 2 -1 3 -1 4 -2
1 -1 3 -1 5 -2
4 -1 5 -2

Exécution de l’algorithme GSP...
>/home/mlee/spmf.jar
=============  Algorithm - STATISTICS =============
 Total time ~ 17 ms
 Frequent sequences count : 20
 Join count : 65
 Max memory (mb):11.511398315429688
Content at file output_SPADE.txt

=== Résultats de SPADE ===
1 -1 #SUP: 4
2 -1 #SUP: 2
3 -1 #SUP: 4
4 -1 #SUP: 4
5 -1 #SUP: 4
4 -1 5 -1 #SUP: 2
3 -1 5 -1 #SUP: 2
1 -1 5 -1 #SUP: 2
3 -1 4 -1 #SUP: 3
2 -1 4 -1 #SUP: 2
1 -1 4 -1 #SUP: 3
2 -1 3 -1 #SUP: 2
1 -1 3 -1 #SUP: 4
1 -1 2 -1 #SUP: 2
2 -1 3 -1 4 -1 #SUP: 2
1 -1 2 -1 3 -1 #SUP: 2
1 -1 2 -1 4 -1 #SUP: 2
1 -1 2 -1 3 -1 4 -1 #SUP: 2
1 -1 3 -1 4 -1 #SUP: 3
1 -1 3 -1 5 -1 #SUP: 2

=== Résultats GSP (propres) ===
<1>  (support: 4)
<2

## EXO 6

In [82]:
import pandas as pd
import subprocess
import re

# =======================
# 1️⃣ Jeu de données
# =======================
data = {
    "S.ID": [10, 20, 30, 40, 50, 60],
    "Sequence": [
        "<{a}{a c}{a d c}>",
        "<{b a}{f b}{a}>",
        "<{a b}{b}{f b}{a e}>",
        "<{a}{a f}{d}>",
        "<{d}{f a c}>",
        "<{a d f}{a e}>"
    ]
}

df = pd.DataFrame(data)
print("=== Jeu de données initial ===")
print(df, "\n")

# =======================
# 2️⃣ Conversion correcte en format SPMF
# =======================
def convert_to_spmf(seq_str):
    seq_str = seq_str.strip()[1:-1]
    itemsets = re.findall(r"\{([^}]*)\}", seq_str)
    spmf_seq = " -1 ".join(itemsets) + " -2"
    return spmf_seq

spmf_sequences = [convert_to_spmf(s) for s in df["Sequence"]]
dataset_str = "\n".join(spmf_sequences)

with open("dataset.txt", "w") as f:
    f.write(dataset_str + "\n")

print("=== Format SPMF généré ===")
print(dataset_str)
print()

# =======================
# 3️⃣ Exécution du GSP (support absolu = 3)
# =======================
total_sequences = len(df)
min_support_absolute = 3
min_support_relative = min_support_absolute / total_sequences

print(f"🎯 PARAMÈTRES CORRIGÉS:")
print(f"   • Support absolu désiré: {min_support_absolute} séquences")
print(f"   • Support relatif à utiliser: {min_support_relative:.3f}")
print(f"   • Calcul: {min_support_absolute} / {total_sequences} = {min_support_relative:.3f}")
print(f"   • En pourcentage: {min_support_relative*100:.1f}%")

command = [
    "java", "-jar", "spmf.jar",
    "run", "SPADE",
    "dataset_numeric.txt", "output_exo6_SPADE.txt",
    str(min_support_relative),  # ⚠️ 0.5 pour 3 séquences
    "false", "10"
]

print("Exécution de l’algorithme SPADE...")
subprocess.run(command)

# =======================
# 4️⃣ Lecture et affichage brut des résultats
# =======================
with open("output_exo6_SPADE.txt", "r") as f:
    results = f.read()

print("=== Résultats de SPADE (bruts) ===")
print(results if results else "Aucun motif fréquent trouvé.")

# =======================
# 5️⃣ Lecture et affichage propre des résultats
# =======================
clean_results = []
with open("output_exo6_SPADE.txt", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("#SUP:")
        seq_part = parts[0].strip()
        support = parts[1].strip() if len(parts) > 1 else "?"
        items = [s.strip() for s in seq_part.split("-1") if s.strip()]
        seq_readable = "".join(f"<{i.strip()}>" for i in items)
        clean_results.append(f"{seq_readable}  (support: {support})")

# =======================
# 6️⃣ Affichage final propre
# =======================
if clean_results:
    print("\n=== Résultats SPADE (propres) ===")
    for seq in clean_results:
        print(seq)
else:
    print("Aucun motif fréquent trouvé.")


=== Jeu de données initial ===
   S.ID              Sequence
0    10     <{a}{a c}{a d c}>
1    20       <{b a}{f b}{a}>
2    30  <{a b}{b}{f b}{a e}>
3    40         <{a}{a f}{d}>
4    50          <{d}{f a c}>
5    60        <{a d f}{a e}> 

=== Format SPMF généré ===
a -1 a c -1 a d c -2
b a -1 f b -1 a -2
a b -1 b -1 f b -1 a e -2
a -1 a f -1 d -2
d -1 f a c -2
a d f -1 a e -2

🎯 PARAMÈTRES CORRIGÉS:
   • Support absolu désiré: 3 séquences
   • Support relatif à utiliser: 0.500
   • Calcul: 3 / 6 = 0.500
   • En pourcentage: 50.0%
Exécution de l’algorithme SPADE...
>/home/mlee/spmf.jar
=============  Algorithm - STATISTICS =============
 Total time ~ 7 ms
 Frequent sequences count : 7
 Join count : 18
 Max memory (mb):10.985214233398438
Content at file output_exo6_SPADE.txt

=== Résultats de SPADE (bruts) ===
1 -1 #SUP: 6
4 -1 #SUP: 4
6 -1 #SUP: 5
1 6 -1 #SUP: 3
1 -1 6 -1 #SUP: 3
6 -1 1 -1 #SUP: 3
1 -1 1 -1 #SUP: 5


=== Résultats SPADE (propres) ===
<1>  (support: 6)
<4>  (support:

In [85]:
import pandas as pd
import subprocess
import re

# =======================
# 1️⃣ Jeu de données
# =======================
data = {
    "S.ID": [10, 20, 30, 40, 50, 60],
    "Sequence": [
        "<{a}{a c}{a d c}>",
        "<{b a}{f b}{a}>",
        "<{a b}{b}{f b}{a e}>",
        "<{a}{a f}{d}>",
        "<{d}{f a c}>",
        "<{a d f}{a e}>"
    ]
}

df = pd.DataFrame(data)
print("=== Jeu de données initial ===")
print(df, "\n")

# =======================
# 2️⃣ Conversion correcte en format SPMF
# =======================
def convert_to_spmf(seq_str):
    seq_str = seq_str.strip()[1:-1]
    itemsets = re.findall(r"\{([^}]*)\}", seq_str)
    spmf_seq = " -1 ".join(itemsets) + " -2"
    return spmf_seq

spmf_sequences = [convert_to_spmf(s) for s in df["Sequence"]]
dataset_str = "\n".join(spmf_sequences)

with open("dataset.txt", "w") as f:
    f.write(dataset_str + "\n")

print("=== Format SPMF généré ===")
print(dataset_str)
print()

# =======================
# 3️⃣ Exécution du GSP (support absolu = 3)
# =======================
total_sequences = len(df)
min_support_absolute = 3
min_support_relative = min_support_absolute / total_sequences

print(f"🎯 PARAMÈTRES CORRIGÉS:")
print(f"   • Support absolu désiré: {min_support_absolute} séquences")
print(f"   • Support relatif à utiliser: {min_support_relative:.3f}")
print(f"   • Calcul: {min_support_absolute} / {total_sequences} = {min_support_relative:.3f}")
print(f"   • En pourcentage: {min_support_relative*100:.1f}%")

command = [
    "java", "-jar", "spmf.jar",
    "run", "GSP",
    "dataset_numeric.txt", "output_exo6_GS.txt",
    str(min_support_relative),  # ⚠️ 0.5 pour 3 séquences
    "false", "10"
]

print("Exécution de l’algorithme GS...")
subprocess.run(command)

# =======================
# 4️⃣ Lecture et affichage brut des résultats
# =======================
with open("output_exo6_GS.txt", "r") as f:
    results = f.read()

print("=== Résultats de GS (bruts) ===")
print(results if results else "Aucun motif fréquent trouvé.")

# =======================
# 5️⃣ Lecture et affichage propre des résultats8
# =======================
clean_results = []
with open("output_exo6_GS.txt", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("#SUP:")
        seq_part = parts[0].strip()
        support = parts[1].strip() if len(parts) > 1 else "?"
        items = [s.strip() for s in seq_part.split("-1") if s.strip()]
        seq_readable = "".join(f"<{i.strip()}>" for i in items)
        clean_results.append(f"{seq_readable}  (support: {support})")

# =======================
# 6️⃣ Affichage final propre
# =======================
if clean_results:
    print("\n=== Résultats GS (propres) ===")
    for seq in clean_results:
        print(seq)
else:
    print("Aucun motif fréquent trouvé.")


=== Jeu de données initial ===
   S.ID              Sequence
0    10     <{a}{a c}{a d c}>
1    20       <{b a}{f b}{a}>
2    30  <{a b}{b}{f b}{a e}>
3    40         <{a}{a f}{d}>
4    50          <{d}{f a c}>
5    60        <{a d f}{a e}> 

=== Format SPMF généré ===
a -1 a c -1 a d c -2
b a -1 f b -1 a -2
a b -1 b -1 f b -1 a e -2
a -1 a f -1 d -2
d -1 f a c -2
a d f -1 a e -2

🎯 PARAMÈTRES CORRIGÉS:
   • Support absolu désiré: 3 séquences
   • Support relatif à utiliser: 0.500
   • Calcul: 3 / 6 = 0.500
   • En pourcentage: 50.0%
Exécution de l’algorithme GS...
>/home/mlee/spmf.jar
=============  Algorithm - STATISTICS =============
 Total time ~ 6 ms
 Frequent sequences count : 4
 Max memory (mb):10.993026733398438

=== Résultats de GS (bruts) ===
1 -1 #SUP: 6
4 -1 #SUP: 4
6 -1 #SUP: 5
1 -1 6 -1 #SUP: 3


=== Résultats GS (propres) ===
<1>  (support: 6)
<4>  (support: 4)
<6>  (support: 5)
<1><6>  (support: 3)
